In [1]:
%load_ext autoreload
%autoreload 1
%aimport 

Modules to reload:


Modules to skip:



In [5]:
from datasets import load_dataset, DatasetDict
from transformers import WhisperProcessor
from datasets import Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from transformers import WhisperForConditionalGeneration
from functools import partial

In [3]:
clean_train100 = load_dataset("openslr/librispeech_asr", "clean", split="train.100")

In [4]:
clean_train360 = load_dataset("openslr/librispeech_asr", "clean", split="train.360")    
clean_valid = load_dataset("openslr/librispeech_asr", "clean", split="validation")
clean_test = load_dataset("openslr/librispeech_asr", "clean", split="test")
other_valid = load_dataset("openslr/librispeech_asr", "other", split="validation")
other_test = load_dataset("openslr/librispeech_asr", "other", split="test")

In [5]:
other_valid[0]["audio"]

In [6]:
print(other_valid['text'][0])
print(other_valid['speaker_id'][0])
other_valid.features


GERAINT AS HE HAD BEEN USED TO DO WHEN HE WAS AT ARTHUR'S COURT FREQUENTED TOURNAMENTS
3660


{'file': Value('string'),
 'audio': Audio(sampling_rate=16000, decode=True, stream_index=None),
 'text': Value('string'),
 'speaker_id': Value('int64'),
 'chapter_id': Value('int64'),
 'id': Value('string')}

In [7]:
audio = other_valid[0]['audio']

In [3]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="english", task="transcribe"
    )

In [18]:
sampling_rate = processor.feature_extractor.sampling_rate
print(sampling_rate)

16000


In [4]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.freeze_encoder()
model.config.use_cache = False
model.generate = partial(
model.generate, language='english', task="transcribe", use_cache=True
    )

In [21]:
model

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [7]:
common_voice = DatasetDict()

common_voice["train"] = load_dataset(
    "united-we-care/United-Syn-Med", split="train"
)
common_voice["test"] = load_dataset(
    "united-we-care/United-Syn-Med", split="test"
)

DatasetNotFoundError: Dataset 'united-we-care/United-Syn-Med' is a gated dataset on the Hub. Visit the dataset page at https://huggingface.co/datasets/united-we-care/United-Syn-Med to ask for access.